In [ ]:
!pip install pyvi
!pip install ftfy
!pip install transformers==4.17.0
!pip install simpletransformers
!pip install vncorenlp
!pip install demoji

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

path_train ='/content/gdrive/MyDrive/2023_Research/JCSCE Response/Dataset/csv/Train_Res.csv'
path_dev ='/content/gdrive/MyDrive/2023_Research/JCSCE Response/Dataset/csv/Dev_Res.csv'
path_test  ='/content/gdrive/MyDrive/2023_Research/JCSCE Response/Dataset/csv/Test_Res.csv'
path_test_original  ='/content/gdrive/MyDrive/2023_Research/JCSCE Response/Dataset/Test.txt'

import argparse


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
from numpy import random
import random as rn
random.seed(12345)
import tensorflow as tf
np.random.seed(12345)
import pandas as pd
rn.seed(12345)
from ftfy import fix_text
from sklearn.utils import shuffle

import os, pickle, re, keras, sklearn, string

In [ ]:
replace_list = {
  'òa': 'oà', 'óa': 'oá', 'ỏa': 'oả', 'õa': 'oã', 'ọa': 'oạ', 'òe': 'oè', 'óe': 'oé','ỏe': 'oẻ',
  'õe': 'oẽ', 'ọe': 'oẹ', 'ùy': 'uỳ', 'úy': 'uý', 'ủy': 'uỷ', 'ũy': 'uỹ','ụy': 'uỵ', 'uả': 'ủa',
  'ả': 'ả', 'ố': 'ố', 'u´': 'ố','ỗ': 'ỗ', 'ồ': 'ồ', 'ổ': 'ổ', 'ấ': 'ấ', 'ẫ': 'ẫ', 'ẩ': 'ẩ',
  'ầ': 'ầ', 'ỏ': 'ỏ', 'ề': 'ề','ễ': 'ễ', 'ắ': 'ắ', 'ủ': 'ủ', 'ế': 'ế', 'ở': 'ở', 'ỉ': 'ỉ',
  'ẻ': 'ẻ', 'àk': u' à ','aˋ': 'à', 'iˋ': 'ì', 'ă´': 'ắ','ử': 'ử', 'e˜': 'ẽ', 'y˜': 'ỹ', 'a´': 'á',

  #Chuẩn hóa 1 số sentiment words/English words
  ':))': '  tích cực ', ':)': ' tích cực ', 'ô kêi': ' ok ', 'okie': ' ok ', ' o kê ': ' ok ',
  'okey': ' ok ', 'ôkê': ' ok ', 'oki': ' ok ', ' oke ':  ' ok ',' okay':' ok ','okê':' ok ',
  ' tks ': u' cám ơn ', 'thks': u' cám ơn ', 'thanks': u' cám ơn ', 'ths': u' cám ơn ', 'thank': u' cám ơn ',
  '⭐': 'star ', '*': 'star ', '🌟': 'star ', '🎉': u' tích cực ',
  'kg ': u' không ','not': u' không ', u' kg ': u' không ', '"k ': u' không ',' kh ':u' không ','kô':u' không ','hok':u' không ',' kp ': u' không phải ',u' kô ': u' không ', '"ko ': u' không ', u' ko ': u' không ', u' k ': u' không ', 'khong': u' không ', u' hok ': u' không ',
  'he he': ' tích cực ','hehe': ' tích cực ','hihi': ' tích cực ', 'haha': ' tích cực ', 'hjhj': ' tích cực ',
  ' lol ': ' tiêu cực ',' cc ': ' tiêu cực ','cute': u' dễ thương ','huhu': ' tiêu cực ', ' vs ': u' với ', 'wa': ' quá ', 'wá': u' quá', 'j': u' gì ', '“': ' ',
  ' sz ': u' cỡ ', 'size': u' cỡ ', u' đx ': u' được ', 'dk': u' được ', 'dc': u' được ', 'đk': u' được ',
  'đc': u' được ','authentic': u' chuẩn chính hãng ',u' aut ': u' chuẩn chính hãng ', u' auth ': u' chuẩn chính hãng ', 'thick': u' tích cực ', 'store': u' cửa hàng ',
  'shop': u' cửa hàng ', 'sp': u' sản phẩm ', 'gud': u' tốt ','god': u' tốt ','wel done':' tốt ', 'good': u' tốt ', 'gút': u' tốt ',
  'sấu': u' xấu ','gut': u' tốt ', u' tot ': u' tốt ', u' nice ': u' tốt ', 'perfect': 'rất tốt', 'bt': u' bình thường ',
  'time': u' thời gian ', 'qá': u' quá ', u' ship ': u' giao hàng ', u' m ': u' mình ', u' mik ': u' mình ',
  'ể': 'ể', 'product': 'sản phẩm', 'quality': 'chất lượng','chat':' chất ', 'excelent': 'hoàn hảo', 'bad': 'tệ','fresh': ' tươi ','sad': ' tệ ',
  'date': u' hạn sử dụng ', 'hsd': u' hạn sử dụng ','quickly': u' nhanh ', 'quick': u' nhanh ','fast': u' nhanh ','delivery': u' giao hàng ',u' síp ': u' giao hàng ',
  'beautiful': u' đẹp tuyệt vời ', u' tl ': u' trả lời ', u' r ': u' rồi ', u' shopE ': u' cửa hàng ',u' order ': u' đặt hàng ',
  'chất lg': u' chất lượng ',u' sd ': u' sử dụng ',u' dt ': u' điện thoại ',u' nt ': u' nhắn tin ',u' tl ': u' trả lời ',u' sài ': u' xài ',u'bjo':u' bao giờ ',
  'thik': u' thích ',u' sop ': u' cửa hàng ', ' fb ': ' facebook ', ' face ': ' facebook ', ' very ': u' rất ',u'quả ng ':u' quảng  ',
  'dep': u' đẹp ',u' xau ': u' xấu ','delicious': u' ngon ', u'hàg': u' hàng ', u'qủa': u' quả ',
  'iu': u' yêu ','fake': u' giả mạo ', 'trl': 'trả lời', '><': u' tích cực ',
  ' por ': u' tệ ',' poor ': u' tệ ', 'ib':u' nhắn tin ', 'rep':u' trả lời ',u'fback':' feedback ','fedback':' feedback ',
  'rolling in the deep': u'món cuốn', '1st impression': u'Ấn tượng đầu tiên', 'fail': u'thất bại',
  'room': u'phòng', 'view': u'quang cảnh', 'member': u'thành viên', 'is the best': u'là tốt nhất', 'check in': u'chụp ảnh',
  'Combo': u'kết hợp', 'staff': u'quản lý', "menu": u"thực đơn", "review": u"nhận xét", "v*i": u"tiêu cực", "mode": u"chế độ", "check": u"kiểm tra",
  "vote": u"bình chọn", "reset": u"làm mới", "bill": u"hoá đơn", "free": u"miễn phí", "out": u"đi ra", "upgrade": u"nâng cấp", "reception": u"lễ tân",
  "rep": u"trả lời", "mix": u"kếp hợp", 'search': u'tìm kiếm', "tone": u'tông', "highly recomended": "nên đề xuất",
  "st impresion": "ấn tượng", "ó": "đó", "ks":u'khách sạn',

  # updated 25/8
  'roling in the dep': 'món cuốn', 'combo': 'kết hợp', 'st impresion': 'ấn tượng', 'm': 'mình',
  'nice': 'tốt', 'ó': 'đó', 'p': 'phút', 'h': 'giờ',
  'tr': 'triệu', 'setup': 'thiết kế', 'take care': 'chăm sóc', 'isue': 'vấn đề',
  'vsinh': 'vệ sinh', 'rip my blader': 'bóng đái tui vỡ rồi', 'i': 'tôi', 'ariport': 'sân bay',
  'bok': 'đặt', 'chil': 'thoải mái', 'desert': 'tráng miệng', 'go': 'đi',
  'this': 'này', 'by chance': 'tình cờ', 'it': 'nó', 'is': 'là',
  'peasant': 'đồng quê', 'fod': 'đồ ăn', 'but': 'nhưng', 'was': 'là',
  'place': 'nơi', 'with': 'với', 'service': 'dịch vụ', 'nodles': 'mì',
  'here': 'ở đây', 'realy': 'rất', 'cheap': 'rẻ', 'gona kep coming back': 'sẽ tiếp tục quay lại',
  'corp': 'cảnh sát', 'station': 'trạm', 'highlight': 'điểm nhấn', 'update': 'cập nhật',
  'deco': 'trang trí', 'seatbelt': 'dây an toàn', 'oder': 'gọi', 'tphcm': 'thành phố hồ chí minh',
  'tour': 'chuyến đi', 't': 'thứ', 'top': 'hạng', 'decor': 'trang trí',
  'say': 'nói', 'late': 'muộn', 'mad': 'giận dữ', 'hot': 'nóng',
  'vnđ': 'vnd', 'đg': 'đường', 'dfoj': 'độ', 'gând': 'gần',
  'tp': 'thành phố', 'max': 'siêu', 'set': 'tập hợp', 'like': 'thích',
  'bm': 'bố mẹ', 'g': 'giờ', 'remote': 'điều khiển', 'wc': 'nhà vệ sinh',
  'zin': 'mới tinh', 'waitres': 'phục vụ', 'gê': 'ghê', 'very': 'rất',
  'highly recomemd': 'rất khuyến khích', 'nhưnh': 'nhưng', 'fen': 'bạn', 'deliver': 'giao hàng',
  'ah': 'à', 'thoy': 'thôi', 'kute': 'dễ thương', 'dng': 'điểm dừng',
  'bos': 'thú cưng', 'auto': 'tự động', 'suite': 'thượng hạng', 'd': 'do',
  'đag': 'đang', 'phats': 'phát', 'recomended': 'khuyến khích', 'vk': 'vợ',
  'ck': 'chồng', 'if': 'nếu', 'ng': 'người', 'zể': 'dễ',
  'ac': 'anh chị', 'confirm': 'xác nhận', 'tuk': 'tức', 'gồn': 'gồm',
  'ap': 'áp', 'nhug': 'nhưng', 'gr': 'nhóm', 'e': 'em',
  'veiw': 'view', 'ik': 'í', 'đer': 'để', 'hn': 'hà nội',
  'overtasted': 'quá vị', 'diet': 'ăn kiêng', 'lf': 'làm', 'healthy': 'sức khoẻ',
  'tc': 'tinh chất', 'bahn': 'bánh', 'what': 'cái', 'thịeu': 'thiệu',
  'ák': 'á', 'blacklist': 'danh sách đen', 'note': 'ghi chú', 'z': 'vậy',
  'qusn': 'quán', 'n': 'ngày', 'roftop': 'mái', 'dzô': 'vô',
  'ngag': 'ngang', 'ún': 'uống', 'sb': 'sài gòn', 'theme': 'chủ đề',
  'cagt': 'công an giao thông', 'trog': 'trong', 'never comeback again': 'không bao giờ quay lại', 'fancy': 'hào nhoáng',
  'get': 'lấy', 'sand': 'sản', 'mk': 'mình', 'parking': 'đỗ xe',
  'and': 'và', 'price': 'giá', 'test': 'thử', 'bv': 'bảo vệ',
  'line': 'hàng', 'uốg': 'uống', 'zễ': 'dễ', 'hoy': 'thôi',
  'ohucj': 'phục', 'sbay': 'sân bay', 'pn': 'bạn', 'ltinh': 'linh tinh',
  'ngoaig': 'ngoài', 'cmk': 'chúng mình', 'u': 'trẻ con', 'come': 'tới',
  'birtbday': 'sinh nhật', 'đh': 'đại học', 'b': 'bạn', 'homemade': 'nhà làm',
  'tpho': 'thành phố', 'team': 'nhóm', 'both': 'cả', 'ráta': 'rất',
  'k': ' ', 'col': 'ngầu', 'quand': 'quán', 'thưingf': 'thường',
  'xún': 'xuống', 'delay': 'trễ', '₫': 'vnd', 'đ': 'vnd',
  'load': 'tải', 'love': 'yêu', 'suport': 'hỗ trợ', 'they': 'họ',
  'rent': 'thuê', 'car': 'xe', 'enthusiasticaly': 'nhiệt tình', 'lovingly': 'dễ thương',
  'extremely': 'cực kỳ', 'went': 'đi', 'times': 'lần', 'rented': 'thuê',
  'motorbike': 'xe máy', 'pleased': 'hài lòng', 'trip': 'chuyến đi', 'al day': 'cả ngày',
  'schedule': 'lịch trình', 'of': 'của', 'more': 'càng', 'without': 'không có',
  'any': 'bất kỳ', 'problems': 'vấn đề', 'surely': 'chắc chắn', 'number': 'số',
  'adres': 'địa chỉ', 'when': 'khi', 'ace': 'anh chị em', 'l': ' ',
  'dthuong': 'dễ thương', 'checkin': 'ghé thăm', 'must': 'phải', 'try': 'thử',
  'nhg': 'nhưng', 'deal': 'đàm phán', 'copy': 'sao chép', 'experiance': 'kinh nghiệm',
  'nh': 'nhưng', 'tue': 'tư', 'tx': 'thị xã', 'sgn': 'sài gòn',
  'anti': 'ghét', 'ord': 'gọi', 'bf': 'ăn sáng', 'quya': 'quay',
  'od': 'gọi', 'rọing': 'rộng', 'uhm': 'ukm', 'membership': 'thành viên',
  'complaint': 'phàn nàn', 'style': 'kiểu', 'nsnd': 'nghệ sĩ nhân dân', 'nsưt': 'nghệ sĩ ưu tú',
  'nc': 'nước', 'sumer': 'mùa hè', 'you': 'bạn', 'for': 'vì',
  'run': 'chạy', 'cn': 'chủ nhật', 'cv': 'công viên', 'met': 'gặp nhau',
  'only': 'chỉ duy nhất', 'best': 'tốt nhất', 'bnhiu': 'bao nhiêu', 'chugn': 'chung',
  'nch': 'nói chuyện', 'take away': 'mang đi', 'thườg': 'thường', 'vieynam': 'vietnam',
  'dtrai': 'đẹp trai', 'vaof': 'vào', 'ns': 'nói', 'delicous': 'ngon',
  'cg': 'cũng', 'nt': 'nhắn tin', 'vn': 'việt nam', 'jo': 'giờ',
  'thcs': 'trung học cơ sở', 'folow': 'theo dõi', 'howph': 'hợp', 'hjh': 'hehe',
  'funy': 'hài hước', 'always': 'luôn luôn', 'carefuly': 'cẩn thận', 'next': 'kế tiếp',
  'tiafn': 'tuần', 'điwsa': 'đứa', 'thíh': 'thích', 'đêuf': 'đều',
  'viawf': 'vừa', 'mh': 'mình', 'đj': 'đi', 'stop': 'dừng lại',
  'thoid': 'thói', 'tiênga': 'tiếng', 'viatnemese': 'vietnamese', 'break': 'khoảng nghỉ',
  'clip': 'video', 'super': 'siêu', 'cskh': 'chăm sóc khách hàng', 'sv': 'sinh viên',
  'ﾟ▽ﾟ': ' ', 'airbnb': 'sân bay', 'lăms': 'lắm', 'nah': 'nha',
  'per one': 'mỗi cái', 'art': 'nghệ thuật', 'ntn': 'như thế nào', 'iá': 'giá',
  'bnhieu': 'bao nhiêu', 'àh': 'à', 'wtf': 'tiêu cực', 'aen': 'ăn',
  'saler': 'nhân viên bán hàng', 'rule': 'thô lỗ', 'cozy': 'ấm áp', 'kbiet': 'không biết',
  'hoi': 'thôi', 'bả': '', 'mxh': 'mạng xã hội', 'nhiwn': 'nhơn',
  'seach': 'tìm', 'trol': 'lừa', 'btw': 'nhân tiện thì', 'godbye': 'tạm biệt',
  'ơe': 'ở', 'days': 'ngày', 'pic': 'ảnh', 'mìh': 'mình',
  'bùi_': 'bùi', 'hs': 'học sinh', 'godluck': 'chúc may mắn', 'group': 'nhóm',
  'zt': 'dễ thương', 'fai': 'phải', 'ad': 'admin', 'hj': 'như',
  'deos': 'đéo', 'situatiuon': 'tình_huống', 'or': 'hoặc', 'bđ': 'bắt đầu',
  'seaview': 'view biển', 'niawx': 'nữa', 'mbh': 'mũ bảo hiểm', 'list': 'danh sách',
  'reviw': 'nhận xét', 'pvu': 'phục vụ', 'wks': 'tuần', 'mem': 'thành viên',
  'aqn': 'ăn', 'resident': 'khu dân cư', 'nhiu': 'nhiêu', 'thíc': 'thích',
  'vcl': 'tiêu cực', 'nhah': 'nhanh', 'fuc': 'phục', 'sunset': 'hoàng hôn',
  'make up': 'trang điểm', 'up': 'tăng', 'many': 'nhiều', 'chalenge': 'thử thách',
  'tromg': 'trong', 'horible': 'kinh khủng', 'pet': 'thú cưng', 'manager': 'quản lý',
  'coa': 'có', 'make': 'làm', 'ày': 'này', 'ak': 'à',
  'live': 'sống', 'heng': 'chen', 'tym': 'tim', 'ngta': 'người ta',
  'min': 'mình', 'godjob': 'tốt lắm', 'disconut': 'giảm giá', 'cin': 'xin',
  'trunwg': 'trưng', 'zô': 'vào', 'tiẹun': 'tiện', 'pas': 'mật khẩu',
  'r': 'rồi', 'cmj': 'tiêu cực', 'bj': 'bao giờ', 'mạnhmojt': 'mạnh một',
  'design': 'thiết kế', 'bđầu': 'bắt đầu', 'quans': 'quán', 'recoment': 'khuyên',
  'nhừn': 'nhưng', 'u_u': 'tiêu cực', 'cacs': 'các', 'hapy': 'vui vẻ',
  'banf': 'bàn', 'nchung': 'nói chung', 'lm': 'làm', 'bonus': 'thêm nữa',
  'cũq': 'cũng', 'owner': 'chủ quán', 'has': 'có', 'quite': 'khá',
  'lot': 'nhiều', 'experience': 'kinh nghiệm', 'making': 'làm', 'had': 'có',
  'prety': 'xinh', 'recipes': 'công thức', 'we': 'chúng tôi', 'ordered': 'gọi',
  'were': 'tất cả', 'great': 'tuyệt', 'dishes': 'bữa ăn', 'employes': 'nhân viên',
  'are': 'là', 'friendly': 'thân thiện', 'welcoming': 'chào mừng', 'made': 'khiến',
  'us': 'chúng tôi', 'fel right': 'cảm thấy như', 'have': 'có', 'litle': 'một ít',
  'french': 'pháp', 'that': 'mà', 'restaurant': 'nhà hàng', 'definitely': 'nhất định',
  'stoping': 'dừng', 'craving': 'thèm', 'bit': 'một chút', 'western': 'phương tây',
  'khoi': 'khỏi', 'mih': 'mình', 'hnay': 'hôm nay', 'inbox': 'nhắn tin',
  'after': 'sau khi', 'credit': 'thanh toán', 'slots': 'chỗ', 'hanghf': 'hàng',
  'ík': 'đấy', 'tgdđ': 'thế giới di động', 'nhỉetj': 'nhiệt', 'tuor': 'tour',
  'kco': 'không có', 'gthieu': 'giới thiệu', 'đubgs': 'đúng', 'favourite': 'yêu thích',
  'dzách': 'đỉnh nhất', 'xug': 'xung', 'rex': '`', 'stres': 'căng thẳng',
  'bsang': 'bữa sáng',

  #dưới 3* quy về 1*, trên 3* quy về 5*
  '6 sao': ' 5star ','6 star': ' 5star ', '5star': ' 5star ','5 sao': ' 5star ','5sao': ' 5star ',
  '4*': '4star', "🤩" : "tốt",
  'starstarstarstarstar': ' 5star ', '1 sao': ' 1star ', '1sao': ' 1star ','2 sao':' 1star ','2sao':' 1star ',
  '2 starstar':' 1star ','1star': ' 1star ', '0 sao': ' 1star ', '0star': ' 1star ',
  "ship": "vận chuyển", "sđt": "số điện thoại", "sdt": "số điện thoại", "đt": "điện thoại",
  "shop": "cửa hàng", "trc": "trước", "bt": "biết", "nge": "nghe", "nhoa": "nha",
  "m": "mình", "o": "ở", "ng": "người", "zai": "trai", "gg": "google",
  "mik": "mình", "gđ": "gia đình",  "mb": "miền bắc",
  "ko": "không", "a": "anh", "r": "rồi", "mn": "mọi người", "rv": "nhận xét",
  "k": " không ", "vs": "với", "zê": "dễ", "thf": "thì", 'phcu5': "phục", "đn": "đà nẵng",
  "kh": "không", "sg": "sài gòn", "oto": "ô tô", "luac" :"lúc", "nàu": "nào", "cx": "cũng",
  "khong": "không", "thù": "thì", "noai": "nói", "bth": "bình thường", "thk": "thích",
  "kg": "không", "ql": "quản lý", "đêr": 'để', "dv": "dịch vụ", "ib": "tin nhắn",
  "khg": "không", "ae": "anh em", "qa": "qua", "cmnd": "chứng minh nhân dân",
  "tl": "trả lời", "ktv": "kế toán viên", "cmt": "chứng minh thư", 'khôg': "không", "khj": "khi",
  "r": "rồi",
  "fb": "mạng xã hội", # facebook
  "face": "mạng xã hội",
  "thanks": "cảm ơn",
  "thank": "cảm ơn",
  "tks": "cảm ơn",
  "tk": "cảm ơn",
  "ok": "tốt", 'okela': "tốt",
  "dc": "được",
  "vs": "với",
  "đt": "điện thoại",
  "thjk": "thích",
  "qá": "quá",
  "trể": "trễ",
  "bgjo": "bao giờ",
  'ksan': 'khách sạn',
  "ncl" : "nói chung",
  "j": "gì",
  "h": "giờ",
  "ô": "ông",
  "đep":"đẹp",
  "4G": "mạng",
  "hdv": "hướng dẫn viên",
  "tưoi":"tươi",
  "wow": "tuyệt vời",
  "nèe": "nè",
  "bik": "biết",
  "nv": "nhân viên",
  "ks": "khách sạn",
  "cf": "cafe",
  "w": "với",
  "qn": "Quy Nhơn",
  "pv": "phóng viên",
  "tp": "thành phố",
  "hcm": "hồ chí minh",
  "&": "và",
  "vc": "việc",
  "dh": "đại học"}

In [ ]:
emoji_list = {
    #Quy các icon về 2 loại emoj: Tích cực hoặc tiêu cực
  "👹": "tiêu cực", "👻": "tích cực", "💃": "tích cực",'🤙': ' tích cực ', '👍': ' tích cực ',
  "💄": "tích cực", "💎": "tích cực", "💩": "tiêu cực","😕": "tiêu cực", "😱": "tiêu cực", "😸": "tích cực",
  "😾": "tiêu cực", "🚫": "tiêu cực",  "🤬": "tiêu cực","🧚": "tích cực", "🧡": "tích cực",'🐶':' tích cực ',
  '👎': ' tiêu cực ', '😣': ' tiêu cực ','✨': ' tích cực ', '❣': ' tích cực ','☀': ' tích cực ',
  '♥': ' tích cực ', '🤩': ' tích cực ', 'like': ' tích cực ', '💌': ' tích cực ',
  '🤣': ' tích cực ', '🖤': ' tích cực ', '🤤': ' tích cực ', ':(': ' tiêu cực ', '😢': ' tiêu cực ',
  '❤': ' tích cực ', '😍': ' tích cực ', '😘': ' tích cực ', '😪': ' tiêu cực ', '😊': ' tích cực ',
  '?': ' ? ', '😁': ' tích cực ', '💖': ' tích cực ', '😟': ' tiêu cực ', '😭': ' tiêu cực ',
  '💯': ' tích cực ', '💗': ' tích cực ', '♡': ' tích cực ', '💜': ' tích cực ', '🤗': ' tích cực ',
  '^^': ' tích cực ', '😨': ' tiêu cực ', '☺': ' tích cực ', '💋': ' tích cực ', '👌': ' tích cực ',
  '😖': ' tiêu cực ', '😀': ' tích cực ', ':((': ' tiêu cực ', '😡': ' tiêu cực ', '😠': ' tiêu cực ',
  '😒': ' tiêu cực ', '🙂': ' tích cực ', '😏': ' tiêu cực ', '😝': ' tích cực ', '😄': ' tích cực ',
  '😙': ' tích cực ', '😤': ' tiêu cực ', '😎': ' tích cực ', '😆': ' tích cực ', '💚': ' tích cực ',
  '✌': ' tích cực ', '💕': ' tích cực ', '😞': ' tiêu cực ', '😓': ' tiêu cực ', '️🆗️': ' tích cực ',
  '😉': ' tích cực ', '😂': ' tích cực ', ':v': '  tích cực ', '=))': '  tích cực ', '😋': ' tích cực ',
  '💓': ' tích cực ', '😐': ' tiêu cực ', ':3': ' tích cực ', '😫': ' tiêu cực ', '😥': ' tiêu cực ',
  '😃': ' tích cực ', '😬': ' tiêu cực ', '😌': ' tiêu cực ', '💛': ' tích cực ', '🤝': ' tích cực ', '🎈': ' tích cực ',
  '😗': ' tích cực ', '🤔': ' tiêu cực ', '😑': ' tiêu cực ', '🔥': ' tiêu cực ', '🙏': ' tiêu cực ',
  '🆗': ' tích cực ', '😻': ' tích cực ', '💙': ' tích cực ', '💟': ' tích cực ',
  '😚': ' tích cực ', '❌': ' tiêu cực ', '👏': ' tích cực ', ';)': ' tích cực ', '<3': ' tích cực ',
  '🌝': ' tích cực ',  '🌷': ' tích cực ', '🌸': ' tích cực ', '🌺': ' tích cực ',
  '🌼': ' tích cực ', '🍓': ' tích cực ', '🐅': ' tích cực ', '🐾': ' tích cực ', '👉': ' tích cực ',
  '💐': ' tích cực ', '💞': ' tích cực ', '💥': ' tích cực ', '💪': ' tích cực ',
  '💰': ' tích cực ',  '😇': ' tích cực ', '😛': ' tích cực ', '😜': ' tích cực ',
  '🙃': ' tích cực ', '🤑': ' tích cực ', '🤪': ' tích cực ','☹': ' tiêu cực ',  '💀': ' tiêu cực ',
  '😔': ' tiêu cực ', '😧': ' tiêu cực ', '😩': ' tiêu cực ', '😰': ' tiêu cực ', '😳': ' tiêu cực ',
  '😵': ' tiêu cực ', '😶': ' tiêu cực ', '🙁': ' tiêu cực ', "☑": "tích cực", "🎍": "tích cực",
  "🍸": "tích cực", "🎉": "tích cực", "🍧": "tích cực", "🚖": "tích cực", "🍽": "tích cực",
  "🍝": "tích cực", "😅": "tiêu cực", "🤨": "tiêu cực", "🍷": "tích cực", "🍛": "tích cực",
  "👤": "tích cực", "🛑": "tiêu cực", "🍒": "tích cực", "🍨": "tích cực", "🍦": "tích cực",
  "☀️": "tích cực", "👎🏻": "tiêu cực", "🍹": "tích cực", "❤️": "tích cực", "⭐": "tích cực",
  "👍🏻": "tích cực", "🍵": "tích cực", "🥰": "tích cực", "🔆": "tích cực", "🙆🏻": "tích cực",
  "💃🏻": "tích cực", "🍶": "tích cực", "🙅🏻‍♀️": "tích cực", "📞": "tích cực", "😽": "tích cực",
  "🍗": "tích cực", "🍺": "tích cực", "📸": "tích cực", "👌🏻": "tích cực", "🍔": "tích cực",
  "💔": "tiêu cực", "🛶": "tích cực", "😻": "tích cực", "🥨": "tích cực", "👌": "tích cực",
  "♀️": "tiêu cực", "🥺": "tiêu cực", "⭕": "tích cực", "♥️": "tích cực", "🏻": "", "☺️": "tích cực"
}

In [ ]:
def tokmap(tok):
    if tok in replace_list:
        return replace_list[tok]

    import demoji

    dict_emoji = demoji.findall(tok)
    dict_emoji = {v: k for k, v in dict_emoji.items()}

    list_emoji_string = demoji.findall_list(tok)

    temp = ""
    for emoji_string in list_emoji_string:
      emoji = dict_emoji[emoji_string]

      if emoji in emoji_list:
        temp += emoji_list[emoji] + " "

    if temp != "":
      return temp

    return tok

remove_words = ["“","”", "…", ". . ."]

def normalText(sent):

    # Xóa các từ đặc biệt
    for item in remove_words:
      sent = sent.replace(item, "")

    sent = sent.replace("\n", " ").replace("\t", " ").strip()
    #Chuẩn hóa tiếng Việt, xử lý emoj, chuẩn hóa tiếng Anh, thuật ngữ
    sent = sent.lower()
    tokens = sent.split(" ")
    tokens = map(tokmap, tokens)
    sent =  " ".join(tokens).strip()
    sent = re.sub('\\s+',' ', sent)

    patPrice = r'([0-9]+k?(\s?-\s?)[0-9]+\s?(k|K))|([0-9]+(.|,)?[0-9]+\s?(triệu|ngàn|trăm|k|K))|([0-9]+k)'
    patHagTag = r'#\s?[aăâbcdđeêghiklmnoôơpqrstuưvxyàằầbcdđèềghìklmnòồờpqrstùừvxỳáắấbcdđéếghíklmnóốớpqrstúứvxýảẳẩbcdđẻểghỉklmnỏổởpqrstủửvxỷạặậbcdđẹệghịklmnọộợpqrstụựvxỵãẵẫbcdđẽễghĩklmnõỗỡpqrstũữvxỹAĂÂBCDĐEÊGHIKLMNOÔƠPQRSTUƯVXYÀẰẦBCDĐÈỀGHÌKLMNÒỒỜPQRSTÙỪVXỲÁẮẤBCDĐÉẾGHÍKLMNÓỐỚPQRSTÚỨVXÝẠẶẬBCDĐẸỆGHỊKLMNỌỘỢPQRSTỤỰVXỴẢẲẨBCDĐẺỂGHỈKLMNỎỔỞPQRSTỦỬVXỶÃẴẪBCDĐẼỄGHĨKLMNÕỖỠPQRSTŨỮVXỸ]+'
    patURL = r"(?:http://|www.)[^\"]+"
    sent = re.sub(patURL,'website',sent)
    sent = re.sub(patHagTag,' hagtag ',sent)
    sent = re.sub(patPrice, ' giá_tiền ', sent)
    sent = re.sub('\.+','.',sent)
    sent = re.sub('(hagtag\\s+)+',' hagtag ',sent)
    sent = re.sub('\\s+',' ',sent)
    return sent

def normalize_elonge_word(sent):
    s_new = ' '
    for char in sent:
       if char != s_new[len(s_new)-1]:
         s_new+=char
    return s_new.strip()


def clean_doc(doc, word_segment=False, lower_case=True):
    doct = fix_text(doc)
    for punc in string.punctuation:
        doc = doc.replace(punc,' '+ punc + ' ')
    doc = re.sub('\\s+',' ',doc)
    doc = normalText(doc)

    # replace số
    doc = re.sub(r"[0-9]+", " num ", doc)
    doc = re.sub(r" num (\s+ num)+", " num ", doc)
    doc = re.sub(r"\s+", " ", doc)

    # Normalize alonge word
    doc = normalize_elonge_word(doc)
    if word_segment == True:
      doc = rdrsegmenter.tokenize(doc)
      doc = ' '.join(doc[0])
      doc = doc.translate(doc.maketrans('', '', string.punctuation.replace("_",""))).replace("giá _ tiền", "giá_tiền").replace("giátiền", "giá_tiền")
    else:
      doc = doc.translate(doc.maketrans('', '', string.punctuation)).replace("giá _ tiền", "giá tiền").replace("giátiền", "giá tiền")

    doc = re.sub('\\s+',' ',doc).strip()
    return doc

In [ ]:
from pyvi import ViTokenizer, ViPosTagger
rdrsegmenter = ViTokenizer
import argparse

In [ ]:
print(clean_doc("người ta có bạn bè nhìn vui thật tưoi.", word_segment=False, lower_case=True))
print(clean_doc("giá 45k giá ngon", word_segment=False, lower_case=True))
print(clean_doc("ai bik", word_segment=False, lower_case=True))

người ta có bạn bè nhìn vui thật tươi
giá giá tiền giá ngon
ai biết


In [ ]:
df_train = pd.read_csv(path_train)
df_train.head()

,Unnamed: 0,X_data,Y_CSC
0,0,"Cả nhà mình đi ăn, hết hơn 1 triệu. Pizza với ...",giá tiền tạm và chất lượng tạm và không gian r...
1,1,"Đúng là chỉ được cái tiếng, đồ ăn thì ít giá t...",lựa chọn rất tệ và giá tiền rất tệ và phục vụ ...
2,2,"Mình ăn set 189k, ấn tượng ko đc tốt lắm: + Ph...",chất lượng rất tệ và phục vụ tệ và giá tiền tạm
3,3,Mỗi lần thèm đồ biển là cả nhà mình đến gió bi...,địa chỉ rất tốt và chất lượng rất tốt và phục ...
4,4,Bố mẹ mình có vẻ thích ăn sen vì gần nhà nhưng...,địa chỉ rất tốt và không gian rất tốt và lựa c...


In [ ]:
# csc task
df_train = pd.read_csv(path_train)
x_train = [clean_doc(item) for item in df_train["X_data"].tolist()]
y_train = [clean_doc(item) for item in df_train["Y_CSC"].tolist()]
print(len(x_train), len(y_train))
prefix = ["csc"]*len(x_train)
df_train_csc = pd.DataFrame(list(zip(x_train, y_train,prefix)),
               columns =['input_text', 'target_text','prefix'])
df_train_csc.head()

6521 6521


,input_text,target_text,prefix
0,cả nhà mình đi ăn hết hơn num triệu piza với p...,giá tiền tạm và chất lượng tạm và không gian r...,csc
1,đúng là chỉ được cái tiếng đồ ăn thì ít giá th...,lựa chọn rất tệ và giá tiền rất tệ và phục vụ ...,csc
2,mình ăn tập hợp giá tiền ấn tượng không được t...,chất lượng rất tệ và phục vụ tệ và giá tiền tạm,csc
3,mỗi lần thèm đồ biển là cả nhà mình đến gió bi...,địa chỉ rất tốt và chất lượng rất tốt và phục ...,csc
4,bố mẹ mình có vẻ thích ăn sen vì gần nhà nhưng...,địa chỉ rất tốt và không gian rất tốt và lựa c...,csc


In [ ]:
# csc task
df_dev = pd.read_csv(path_dev)
x_dev = [clean_doc(item) for item in df_dev["X_data"].tolist()]
y_dev = [clean_doc(item) for item in df_dev["Y_CSC"].tolist()]
print(len(x_dev), len(y_dev))
prefix = ["csc"]*len(x_dev)
df_dev_csc = pd.DataFrame(list(zip(x_dev, y_dev,prefix)),
               columns =['input_text', 'target_text','prefix'])
df_dev_csc.head()

976 976


,input_text,target_text,prefix
0,quán là papa chicken nên món nào cũng toàn gà ...,chất lượng tốt và không gian rất tốt và phục v...,csc
1,mình đặt qua now cơm chiên có mùi như bị khét ...,giá tiền tạm và lựa chọn rất tệ,csc
2,ngày xưa bún bò ở nét huế là số num giờ thì nh...,giá tiền tệ và chất lượng tốt và phục vụ tệ,csc
3,ăn quán cũng đã lâu nhưng giờ ít khi ghé trừ k...,chất lượng tạm và không gian tốt và vấn đề khá...,csc
4,không gian đẹp thoáng mát rất thích tích cực k...,không gian rất tốt và chất lượng rất tốt và ph...,csc


In [ ]:
# csc task
df_test = pd.read_csv(path_test)
x_test = [clean_doc(item) for item in df_test["X_data"].tolist()]
y_test = [clean_doc(item) for item in df_test["Y_CSC"].tolist()]
print(len(x_test), len(y_test))
prefix = ["csc"]*len(x_test)
df_test_csc = pd.DataFrame(list(zip(x_test, y_test,prefix)),
               columns =['input_text', 'target_text','prefix'])
df_test_csc.head()

1821 1821


,input_text,target_text,prefix
0,quầy bufet đa dạng quán ngay mặt tiền và rất g...,lựa chọn rất tốt và địa chỉ rất tốt và chất lư...,csc
1,không gian quán hơi nhỏ nhưng thiết kế tạo cảm...,không gian tạm và phục vụ rất tốt và chất lượn...,csc
2,trước giờ ghé phố num chỉ ăn mâm cơm việt hôm ...,không gian rất tốt và chất lượng tốt và địa ch...,csc
3,quán nằm gần emart vừa quẹo từ phạm văn đồng v...,địa chỉ rất tốt và không gian tệ và chất lượng...,csc
4,rất là vui đồ ăn ngon đồ uống lạ miệng rất thí...,chất lượng rất tốt và lựa chọn rất tốt và phục...,csc


In [ ]:
df_train = df_train_csc
df_dev = df_dev_csc
df_test = df_test_csc
print(len(df_train),len(df_test))
df_test.head()

6521 1821


,input_text,target_text,prefix
0,quầy bufet đa dạng quán ngay mặt tiền và rất g...,lựa chọn rất tốt và địa chỉ rất tốt và chất lư...,csc
1,không gian quán hơi nhỏ nhưng thiết kế tạo cảm...,không gian tạm và phục vụ rất tốt và chất lượn...,csc
2,trước giờ ghé phố num chỉ ăn mâm cơm việt hôm ...,không gian rất tốt và chất lượng tốt và địa ch...,csc
3,quán nằm gần emart vừa quẹo từ phạm văn đồng v...,địa chỉ rất tốt và không gian tệ và chất lượng...,csc
4,rất là vui đồ ăn ngon đồ uống lạ miệng rất thí...,chất lượng rất tốt và lựa chọn rất tốt và phục...,csc


In [ ]:
from simpletransformers.t5 import T5Model, T5Args

train_df = df_train
eval_df = df_dev

model_args = T5Args()
model_args.manual_seed = 42
model_args.max_seq_length = 256
model_args.train_batch_size = 4
model_args.max_length = 150
model_args.learning_rate =  2e-5 #2e-5
model_args.num_train_epochs = 20
model_args.use_multiprocessing = False
model_args.fp16 = False
model_args.num_beams = 4
model_args.no_save = True
model_args.save_steps = -1
model_args.save_eval_checkpoints = False
model_args.save_model_every_epoch = False
model_args.no_cache = True
model_args.save_steps = -1
model_args.reprocess_input_data = False
model_args.overwrite_output_dir = True
model_args.eval_batch_size = 4
model_args.preprocess_inputs = False
model_args.use_early_stopping = False
model_args.early_stopping_delta = 0.01
model_args.early_stopping_metric_minimize = False
model_args.early_stopping_patience = 5

#model = T5Model("mt5", "google/mt5-base", args=model_args)
#model = T5Model("mt5", "google/mt5-large", args=model_args)

model = T5Model("t5", "VietAI/vit5-base", args=model_args)
#model = T5Model("t5", "VietAI/vit5-large", args=model_args)

# Train the model
model.train_model(train_df, eval_data=eval_df)

# Optional: Evaluate the model. We'll test it properly anyway.
results = model.eval_model(eval_df, verbose=True)

Downloading:   0%|          | 0.00/702 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/862M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/801k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.15k [00:00<?, ?B/s]

...

Running Epoch 17 of 20:   0%|          | 0/1631 [00:00<?, ?it/s]

Running Epoch 18 of 20:   0%|          | 0/1631 [00:00<?, ?it/s]

Running Epoch 19 of 20:   0%|          | 0/1631 [00:00<?, ?it/s]

  0%|          | 0/976 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/244 [00:00<?, ?it/s]

In [ ]:
print(results)

{'eval_loss': 0.6170244645766273}


In [ ]:
print(df_test_csc.head())
X_test = df_test_csc["input_text"].tolist()
X_test = ["csc: " + item for item in X_test]
y_test = df_test_csc["target_text"].tolist()

                                          input_text  \
0  quầy bufet đa dạng quán ngay mặt tiền và rất g...   
1  không gian quán hơi nhỏ nhưng thiết kế tạo cảm...   
2  trước giờ ghé phố num chỉ ăn mâm cơm việt hôm ...   
3  quán nằm gần emart vừa quẹo từ phạm văn đồng v...   
4  rất là vui đồ ăn ngon đồ uống lạ miệng rất thí...   

                                         target_text prefix  
0  lựa chọn rất tốt và địa chỉ rất tốt và chất lư...    csc  
1  không gian tạm và phục vụ rất tốt và chất lượn...    csc  
2  không gian rất tốt và chất lượng tốt và địa ch...    csc  
3  địa chỉ rất tốt và không gian tệ và chất lượng...    csc  
4  chất lượng rất tốt và lựa chọn rất tốt và phục...    csc  


In [ ]:
# Predict
preds = model.predict(X_test)
#print(preds)
#print(y_test)

Generating outputs:   0%|          | 0/456 [00:00<?, ?it/s]

/usr/local/lib/python3.9/dist-packages/transformers/tokenization_utils_base.py:3524: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and the tokenizer under the `as_target_tokenizer` context manager to prepare
your targets.

Here is a short example:

model_inputs = tokenizer(src_texts, ...)
with tokenizer.as_target_tokenizer():
    labels = tokenizer(tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)


Decoding outputs:   0%|          | 0/1821 [00:00<?, ?it/s]

In [ ]:
print(preds)
print(y_test)

['chất lượng tạm và địa chỉ rất tốt và lựa chọn rất tốt', 'không gian tạm và phục vụ rất tốt và chất lượng rất tốt và vấn đề khác tốt', 'chất lượng tốt và địa chỉ rất tệ và không gian rất tốt và vấn đề khác tệ', 'địa chỉ tốt và không gian rất tệ và chất lượng tốt và giá tiền tạm', 'chất lượng rất tốt và phục vụ rất tốt và lựa chọn rất tốt và vấn đề khác tệ', 'chất lượng tốt và giá tiền tệ và không gian tốt và phục vụ tệ', 'địa chỉ rất tốt và không gian rất tốt và chất lượng rất tốt và phục vụ rất tốt và vấn đề khác rất tốt', 'chất lượng rất tốt và lựa chọn tạm và không gian rất tốt và địa chỉ tệ', 'chất lượng rất tốt và giá tiền rất tốt và vấn đề khác rất tốt', 'chất lượng tốt và giá tiền tệ và không gian rất tốt và phục vụ rất tốt và lựa chọn rất tốt', 'địa chỉ tạm và chất lượng rất tốt và lựa chọn rất tốt và không gian tạm và vấn đề khác tệ', 'không gian rất tốt và chất lượng tốt và giá tiền rất tốt', 'địa chỉ rất tốt và chất lượng rất tốt và phục vụ tốt và giá tiền tốt và vấn đề khá

In [ ]:
import re
import sys

def get_labels_from_filename(filename):
    labels = []
    with open(filename, 'r', encoding = 'utf-8') as file:
        datasets = file.read()
        count = 0
        for line in datasets.split('\n'):
            if line != '':
                if count == 0:
                    count += 1
                elif count == 1:
                    count += 1
                elif count == 2:
                    labels.append(line.strip())
                    count = 0
        file.close()
    #print(len(labels))
    return labels

def clean_label(label):
    label = re.sub('[^A-Za-z#&]', '', label)
    label = re.sub('\\s+', ' ', label)
    return label

def convert_labels_to_dict(labels):
    dict_labels = []
    for label in labels:
        label_line = label.split('},')
        _dict = {}
        for objectLabel in label_line:
            aspect = clean_label(objectLabel.split(',')[0]).strip()
            polarity = clean_label(objectLabel.split(',')[1]).strip()
            _dict[aspect] = polarity
        dict_labels.append(_dict)
    return dict_labels

def get_common_attributeEntities(dict_labels):
    AttributeEntities = []
    for _dict in dict_labels:
        for key in _dict:
            if key not in AttributeEntities:
                AttributeEntities.append(key)
    AttributeEntities = sorted(AttributeEntities)
    return AttributeEntities

def get_aspects(dict_labels):
    aspects = []
    for _dict in dict_labels:
        for key in _dict:
            aspects.append(key)
    return aspects

def count_aspects(labels, Common_AttributeEntities):
    aspects = get_aspects(labels)
    num_aspects = [0] * len(Common_AttributeEntities)
    for aspect in aspects:
        num_aspects[Common_AttributeEntities.index(aspect)] += 1
    return num_aspects

def evaluation_labels(gold_labels, answer_labels, Common_AttributeEntities):
    num_aspect_gold = count_aspects(gold_labels, Common_AttributeEntities)
    num_aspect_answer = count_aspects(answer_labels, Common_AttributeEntities)
    correct_answer_aspects = [0] * len(Common_AttributeEntities)
    correct_answer_labels = [0] * len(Common_AttributeEntities)

    for i, _dict in enumerate(answer_labels):
        for key in _dict:
            if key in gold_labels[i].keys():
                correct_answer_aspects[Common_AttributeEntities.index(key)] += 1
                if answer_labels[i][key].strip() == gold_labels[i][key].strip():
                    correct_answer_labels[Common_AttributeEntities.index(key)] += 1
    #print('Correct Answer Aspects: ', correct_answer_aspects)
    #print('---------------------------------------------------')
    #print('Correct Answer Labels: ', correct_answer_labels)
    #print('---------------------------------------------------')
    #infor_evaluation(correct_answer_aspects, num_aspect_answer, num_aspect_gold, Common_AttributeEntities)
    #print('---------------------------------------------------')
    infor_evaluation(correct_answer_labels, num_aspect_answer, num_aspect_gold, Common_AttributeEntities)

def infor_evaluation(correct_answer, num_aspect_answer, num_aspect_gold, Common_AttributeEntities):
    for aspect in Common_AttributeEntities:
        if correct_answer[Common_AttributeEntities.index(aspect)] == 0:
            p = r = f = 0.0
        else:
            p = correct_answer[Common_AttributeEntities.index(aspect)] * 100 / num_aspect_answer[Common_AttributeEntities.index(aspect)]
            r = correct_answer[Common_AttributeEntities.index(aspect)] * 100 / num_aspect_gold[Common_AttributeEntities.index(aspect)]
            f = 2 * p * r / (p + r)
        print(aspect)
        print('%0.2f\t%0.2f\t%0.2f' % (p, r, f))
    p = sum(correct_answer) * 100 / sum(num_aspect_answer)
    r = sum(correct_answer) * 100 / sum(num_aspect_gold)
    f = 2 * p * r / (p + r)
    print('-------------------------------------------------------------')
    print('-------------------------------------------------------------')
    print('Mean Precision score: ', round(p,2))
    print('Mean Recall score: ', round(r,2))
    print('Mean F1 score: ', round(f,2))
    print('-------------------------------------------------------------')
    print('-------------------------------------------------------------')

def evaluation_system(gold_labels, answer_labels):
    gold_dicts = convert_labels_to_dict(gold_labels)
    answer_dicts = convert_labels_to_dict(answer_labels)
    AttributeEntities = get_common_attributeEntities(gold_dicts)
    #print('---------------INFORMATION FILE--------------------')
    #print('Aspect Name: ', AttributeEntities)
    #print("Aspect Gold: ", count_aspects(gold_dicts, AttributeEntities))
    #print("Aspect Answer: ", count_aspects(answer_dicts, AttributeEntities))
    #print('---------------------------------------------------')
    evaluation_labels(gold_dicts, answer_dicts, AttributeEntities)

def evaluation_system_by_file(file_gold, file_predict):
    gold_labels = get_labels_from_filename(file_gold)
    answer_labels = get_labels_from_filename(file_predict)
    evaluation_system(gold_labels, answer_labels)


In [ ]:
listLabel = 'PRICES,QUALITY,AMBIENCE,STYLE_OPTIONS,SERVICE,LOCATION,MISCELLANEOUS'
categories = listLabel.split(',')
natural_categories = 'giá tiền,chất lượng,không gian,lựa chọn,phục vụ,địa chỉ,vấn đề khác'
meaning_abstract_list = natural_categories.split(",")

In [ ]:
textPrint =  ""
count = 0
for index,pred in enumerate(preds):
  s = ""
  for category in pred.split(" và "):
      category = category.strip()
      for i,abstract in enumerate(meaning_abstract_list):
        if abstract.strip() in category:
            if "tệ" in category:
              s+= '{' + str(categories[i]) + ', negative}, '
            elif "tốt" in category:
              s+= '{' + str(categories[i]) + ', positive}, '
            elif "tạm" in category:
              s+= '{' + str(categories[i]) + ', neutral}, '
            elif "rất tốt" in category:
              s+= '{' + str(categories[i]) + ', very_positive}, '
            elif "rất tệ" in category:
              s+= '{' + str(categories[i]) + ', very_negative}, '
            else:
               print("error: ", category)
  if s == "":
    print(pred)
    s = "{MISCELLANEOUS, neutral}, "
  textPrint += '#' + str(count +1)+'\n'
  textPrint += X_test[index] + '\n'
  textPrint += s[:len(s)-2] +'\n\n'
  #print("pred: " + pred)
  #print("label: " + s[:len(s)-2])
  #print()
  count +=1

textPrint = textPrint[:-2]
with open('output.txt','w',encoding = 'utf8') as file:
    file.write(textPrint)

In [ ]:
textPrint =  ""
count = 0
for index,pred in enumerate(y_test):
  s = ""
  for category in pred.split(" và "):
      category = category.strip()
      for i,abstract in enumerate(meaning_abstract_list):
        if abstract.strip() in category:
            if "tệ" in category:
              s+= '{' + str(categories[i]) + ', negative}, '
            elif "tốt" in category:
              s+= '{' + str(categories[i]) + ', positive}, '
            elif "tạm" in category:
              s+= '{' + str(categories[i]) + ', neutral}, '
            elif "rất tốt" in category:
              s+= '{' + str(categories[i]) + ', very_positive}, '
            elif "rất tệ" in category:
              s+= '{' + str(categories[i]) + ', very_negative}, '
            else:
               print("error: ", category)
  textPrint += '#' + str(count +1)+'\n'
  textPrint += X_test[index] + '\n'
  textPrint += s[:len(s)-2] +'\n\n'
  #print("pred: " + pred)
  #print("label: " + s[:len(s)-2])
  #print()
  count +=1

textPrint = textPrint[:-2]
with open('y_test.txt','w',encoding = 'utf8') as file:
    file.write(textPrint)

In [ ]:
evaluation_system_by_file("y_test.txt", "output.txt")

AMBIENCE
91.41	89.73	90.56
LOCATION
81.37	78.78	80.05
MISCELLANEOUS
68.22	74.34	71.15
PRICES
84.69	85.66	85.17
QUALITY
88.43	89.06	88.74
SERVICE
90.65	91.19	90.92
STYLEOPTIONS
78.44	79.22	78.83
-------------------------------------------------------------
-------------------------------------------------------------
Mean Precision score:  85.46
Mean Recall score:  86.14
Mean F1 score:  85.8
-------------------------------------------------------------
-------------------------------------------------------------


In [ ]:

path_save = '/content/drive/MyDrive/2023_Research/'
name_file = "output.txt"
!cp "$name_file" "$path_save"